In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [ ]:
data = np.load(
    "../data/sloshing_data.npz"
)

X = data["X"]
alpha = data["alpha_eau"]
U = data["U"]
times = data["time"]

Fx = data["Fx"]
Fy = data["Fy"]
Mz = data["Mz"]

print("Dataset chargé")
print("----------------")
print("time  :", times.shape)
print("X     :", X.shape)
print("alpha :", alpha.shape)
print("U     :", U.shape)
print("Fx    :", Fx.shape)

In [ ]:
displacement = X[:, :, 0] - X[0, :, 0]

XR = np.mean(
    displacement,
    axis=1
)

print("XR shape :", XR.shape)

print(
    f"Déplacement min : {XR.min():.6e} m"
)

print(
    f"Déplacement max : {XR.max():.6e} m"
)

In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(
    times,
    XR,
)

plt.xlabel("t [s]")
plt.ylabel("$X_R$ [m]")
plt.title("Déplacement estimé du réservoir")

plt.grid(True)
plt.show()

In [ ]:
VR = np.gradient(
    XR,
    times
)

AR = np.gradient(
    VR,
    times
)

print(
    f"Vitesse max : {np.max(np.abs(VR)):.6e} m/s"
)

print(
    f"Accélération max : {np.max(np.abs(AR)):.6e} m/s²"
)

In [ ]:
X_relative = X.copy()

X_relative[:, :, 0] = (
    X[:, :, 0]
    - XR[:, None]
)

X_relative[:, :, 1] = X[:, :, 1]
X_relative[:, :, 2] = X[:, :, 2]

In [ ]:
fig, ax = plt.subplots(
    figsize=(14, 5)
)

points0 = X_relative[0, :, :2]

scatter = ax.scatter(
    points0[:, 0],
    points0[:, 1],
    c=alpha[0],
    vmin=0,
    vmax=1,
    s=10,
)

ax.set_aspect("equal")

ax.set_xlabel(r"$\xi$ [m]")
ax.set_ylabel(r"$\eta$ [m]")

ax.set_xlim(
    X_relative[:, :, 0].min(),
    X_relative[:, :, 0].max()
)

ax.set_ylim(
    X_relative[:, :, 1].min(),
    X_relative[:, :, 1].max()
)

title = ax.set_title(
    f"t = {times[0]:.4f} s"
)

In [ ]:
def update(i):

    points = X_relative[i, :, :2]

    scatter.set_offsets(points)

    scatter.set_array(
        alpha[i]
    )

    title.set_text(
        f"Référentiel réservoir — "
        f"t = {times[i]:.4f} s"
    )

    return scatter, title


animation = FuncAnimation(
    fig,
    update,
    frames=len(times),
    interval=30,
    blit=True,
)

plt.close(fig)

HTML(
    animation.to_jshtml()
)

In [ ]:
i = len(times) // 2

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)

# ------------------------------------------------------------
# Référentiel laboratoire
# ------------------------------------------------------------

axes[0].scatter(
    X[i, :, 0],
    X[i, :, 1],
    c=alpha[i],
    vmin=0,
    vmax=1,
    s=10,
)

axes[0].set_title(
    f"Référentiel laboratoire — t={times[i]:.3f}s"
)

axes[0].set_xlabel("x [m]")
axes[0].set_ylabel("y [m]")
axes[0].set_aspect("equal")


# ------------------------------------------------------------
# Référentiel réservoir
# ------------------------------------------------------------

axes[1].scatter(
    X_relative[i, :, 0],
    X_relative[i, :, 1],
    c=alpha[i],
    vmin=0,
    vmax=1,
    s=10,
)

axes[1].set_title(
    "Référentiel réservoir"
)

axes[1].set_xlabel(r"$\xi$ [m]")
axes[1].set_ylabel(r"$\eta$ [m]")
axes[1].set_aspect("equal")


plt.tight_layout()
plt.show()

In [ ]:
np.savez_compressed(
    "../data/sloshing_relative.npz",

    time=times,

    X=X_relative,

    U=U,
    alpha_eau=alpha,

    p_rgh=data["p_rgh"],
    k=data["k"],
    epsilon=data["epsilon"],

    XR=XR,
    VR=VR,
    AR=AR,

    Fx=Fx,
    Fy=Fy,
    Mz=Mz,
)